# Decision metrics for the ADBench baselines

Reads the baseline score files written by `adbench_baselines_colab.ipynb` (`MyDrive/STRAP_ADBench/baseline_scores/`) and computes, for every method, dataset, protocol and seed, the F1, precision, recall and false positive rate of these label-free decision rules:

- `pyod_default`: PyOD's default contamination of 10%, i.e. threshold at the 90th percentile of the training scores (what `predict()` does).
- `quantile_a0.05`, `quantile_a0.1`: threshold at the (1−α) quantile of the training scores, the same nominal false positive rate as STRAP at α.

and two rules that use test labels, reported only as upper bounds:
- `true_contamination`: flag the top-k test samples, k = number of test anomalies.
- `oracle_best_f1`: best F1 over all thresholds.

CPU only, a few minutes. Output: `MyDrive/STRAP_ADBench/baselines_decisions_per_seed.csv` (small; upload it to the chat).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, numpy as np, pandas as pd
from sklearn.metrics import precision_recall_curve
DRIVE_ROOT = '/content/drive/MyDrive/STRAP_ADBench'
IN_DIR = os.path.join(DRIVE_ROOT, 'baseline_scores')
OUT = os.path.join(DRIVE_ROOT, 'baselines_decisions_per_seed.csv')

In [ ]:
def clean(s):
    s = np.asarray(s, float).copy(); f = np.isfinite(s)
    hi, lo = (s[f].max(), s[f].min()) if f.any() else (0.0, 0.0)
    s[np.isposinf(s)] = hi + 1; s[np.isneginf(s) | np.isnan(s)] = lo - 1
    return s

def prf(y, D):
    tp = int((D & (y == 1)).sum()); fp = int((D & (y == 0)).sum())
    p = tp / max(1, int(D.sum())); r = tp / max(1, int(y.sum()))
    return {'f1': 0.0 if p + r == 0 else 2 * p * r / (p + r), 'precision': p, 'recall': r,
            'fpr': fp / max(1, int((y == 0).sum()))}

rows = []
files = sorted(f for f in os.listdir(IN_DIR) if f.endswith('.npz'))
for i, f in enumerate(files):
    ds, protocol, seed = f[:-4].split('__')
    z = np.load(os.path.join(IN_DIR, f)); y = z['y_test'].astype(int)
    methods = sorted({k.split('__')[0] for k in z.files if k.endswith('__test')})
    for m in methods:
        tr, te = clean(z[f'{m}__train']), clean(z[f'{m}__test'])
        base = {'protocol': protocol, 'dataset': ds, 'seed': int(seed[4:]), 'method': m}
        for rule, thr in [('pyod_default', np.quantile(tr, 0.90)),
                          ('quantile_a0.05', np.quantile(tr, 0.95)),
                          ('quantile_a0.1', np.quantile(tr, 0.90))]:
            rows.append({**base, 'rule': rule, **prf(y, te > thr)})
        k = int(y.sum()); top = np.zeros_like(y, bool); top[np.argsort(-te)[:k]] = True
        rows.append({**base, 'rule': 'true_contamination', **prf(y, top)})
        p, r, _ = precision_recall_curve(y, te)
        rows.append({**base, 'rule': 'oracle_best_f1', 'f1': float(np.max(2 * p * r / np.maximum(p + r, 1e-12))),
                     'precision': np.nan, 'recall': np.nan, 'fpr': np.nan})
    if (i + 1) % 50 == 0: print(f'{i+1}/{len(files)} files')
res = pd.DataFrame(rows); res.to_csv(OUT, index=False)
print('saved', OUT, res.shape)
print((100 * res.groupby(['protocol', 'rule', 'method']).f1.mean().unstack('rule')).round(1).to_string())